In [8]:
import pandas as pd
import numpy as np
import zipfile
from sentence_transformers import SentenceTransformer
import torch
import faiss
import sqlite3

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [12]:
def run_sql(db_path, query, params=None, many=False, commit=False, fetch=False):
    """Open a sqlite connection, run a query, and optionally commit/fetch results."""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        if many:
            cursor.executemany(query, params)
        else:
            cursor.execute(query, params or ())
        if commit:
            conn.commit()
        if fetch:
            return cursor.fetchall()


def encode_texts(model, texts, device, batch_size=256, show_progress_bar=True):
    """Encode a list of texts into normalized embeddings using a SentenceTransformer model."""
    return model.encode(
        texts,
        device=device,
        batch_size=batch_size,
        show_progress_bar=show_progress_bar,
        normalize_embeddings=True  # -> Inner product will be cosine similarity
    )


def build_faiss_index(embeddings):
    """Build a FAISS inner-product index from an array of embeddings."""
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index


def search(model, index, df, device, query, k=5, title_col="title"):
    """Embed a query, find its top-k nearest neighbors in a FAISS index, and print the matching rows."""
    query_vec = encode_texts(model, [query], device, batch_size=1, show_progress_bar=False)
    scores, positions = index.search(query_vec, k=k)
    for pos, score in zip(positions[0], scores[0]):
        print(f"id={pos}  score={score:.4f}  title={df.iloc[pos][title_col]}")

<h2>Wikihow SQLite DB</h2>

In [17]:
# extract zip file and read data into df
with zipfile.ZipFile("Compressed_Data/wikihow-cleaned.zip", 'r') as zip_ref:
    zip_ref.extractall("Data")

wikihow_df = pd.read_csv("Data/wikihow-cleaned/wikihow-cleaned.csv")

In [18]:
# only some nans for text
wikihow_df.isna().sum()

summary      0
title        0
text       401
dtype: int64

In [ ]:
# create the table
run_sql("wikihow.db", """
    CREATE TABLE wikihow_articles (
    id int PRIMARY KEY,
    summary varchar,
    title varchar,
    text varchar
    )
""", commit=True)

In [ ]:
# insert the wikihow data
rows_to_append = list(wikihow_df[["summary", "title", "text"]].itertuples(name=None))
run_sql("wikihow.db", "INSERT INTO wikihow_articles VALUES (?, ?, ?, ?)", rows_to_append, many=True, commit=True)

In [19]:
# example query to show it worked
response = run_sql("wikihow.db", "SELECT * FROM wikihow_articles LIMIT 2", fetch=True)
response

[(0,
  'keep related supplies in the same area . make an effort to clean a dedicated workspace after every session . place loose supplies in large , clearly visible containers . use clotheslines and clips to hang sketches , photos , and reference material . use every inch of the room for storage , especially vertical space . use chalkboard paint to make space for drafting ideas right on the walls . purchase a label maker to make your organization strategy semi - permanent . make a habit of throwing out old , excess , or useless stuff each month .',
  'how to be an organized artist 1',
  'if youre a photographer , keep all the necessary lens , cords , and batteries in the same quadrant of your home or studio . paints should be kept with brushes , cleaner , and canvas , print supplies should be by the ink , etc . make broader groups and areas for your supplies to make finding them easier , limiting your search to a much smaller area . some ideas include essential supplies area - - the th

In [20]:
# compare length of db to length of df
response = run_sql("wikihow.db", "SELECT COUNT(*) FROM wikihow_articles", fetch=True)
print(response)
print(len(wikihow_df))

[(214293,)]
214293


<h2>Embeddings and FAISS</h2>

<h3>WikiHow</h3>

In [28]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# Generate embeddings
title_embed = encode_texts(embedding_model, wikihow_df["title"].to_numpy(), device)
title_summary_embed = encode_texts(embedding_model, (wikihow_df["title"] + ". " + wikihow_df["summary"]).to_numpy(), device)

In [ ]:
# Save raw embeddings in numpy files
np.save("Data/Embeddings/title_embed.npy", title_embed)
np.save("Data/Embeddings/title_summary_embed.npy", title_summary_embed)

In [ ]:
# Load in embeddings
title_embed        = np.load("Data/Embeddings/title_embed.npy")
title_summary_embed = np.load("Data/Embeddings/title_summary_embed.npy")

In [ ]:
# Build FAISS indices
# each FAISS index maps back to the same id in the db/wikihow df since they were both generated in the same order
title_index    = build_faiss_index(title_embed)
combined_index = build_faiss_index(title_summary_embed)

In [ ]:
# Save
faiss.write_index(title_index,    "title_index.faiss")
faiss.write_index(combined_index, "combined_index.faiss")

In [ ]:
# Compress the FAISS indices into a zip
with zipfile.ZipFile("Compressed_Data/wikihow_faiss_indices.zip", 'w') as zip_ref:
    zip_ref.write("title_index.faiss",    "title_index.faiss")
    zip_ref.write("combined_index.faiss", "combined_index.faiss")

In [ ]:
# Unpack FAISS indices
with zipfile.ZipFile("Compressed_Data/wikihow_faiss_indices.zip", 'r') as zip_ref:
    zip_ref.extractall("Data/Embeddings")

In [23]:
# Load FAISS indices
title_index    = faiss.read_index("Data/Embeddings/title_index.faiss")
combined_index = faiss.read_index("Data/Embeddings/combined_index.faiss")

In [ ]:
# Test query
search(embedding_model, title_index, wikihow_df, device, "how to be chill")

id=64769  score=1.0000  title=how to be chill
id=154582  score=0.8396  title=how to chill 1
id=154583  score=0.8296  title=how to chill 2
id=154584  score=0.8022  title=how to chill 3
id=163692  score=0.7354  title=how to give someone the chills 1


## HowTo100M Video Preprocessing

In [2]:
# Task-2-specific imports (numpy, torch, faiss and `device` come from the cells above)
import cv2                                   # OpenCV: reading video frames
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
import os

# --- clip-generation settings ---
SCAN_FPS         = 1.0     # how many frames per second we look at while scanning a video
SCENE_THRESHOLD  = 0.5     # colour-histogram distance above which a new clip starts
MAX_CLIP_SECONDS = 15.0    # also close a clip once it reaches this length
FRAMES_PER_CLIP  = 3       # number of keyframes averaged into one clip embedding

# --- model + dataset location ---
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
BUNDLE_DIR = "Data/HowTo100M"            # folder with the two .csv files
VIDEO_DIR  = "Data/HowTo100M/videos"     # folder with the <video_id>.mp4 files

In [11]:
# extract zip file
with zipfile.ZipFile("Compressed_Data/howto100m_bundle.zip", 'r') as zip_ref:
    zip_ref.extractall("Data")

In [5]:
av_vids_df = pd.read_csv(BUNDLE_DIR + "/available_videos.csv")
task_ids_df = pd.read_csv(BUNDLE_DIR + "/task_ids.csv", sep="\t", header=None, names=["task_id", "title"])

display(task_ids_df)
display(av_vids_df)

,task_id,title
0,0,Make a Mexican Bean Toast
1,1,Make a Cinnamon Toast Sandwich
2,2,Make Avocado on Toast
3,3,Make Baked Peach French Toast
4,4,Make Buttered Toast
...,...,...
123515,123515,Learn the Capitals of the World
123516,123516,Prepare for the Geography Bee
123517,123517,Read a Nautical Chart
123518,123518,Remember the Seven Continents


,video_id,category_1,category_2,rank,task_id
0,8XaqCvOpKxs,Education and Communications,Subjects,10,47861
1,cfdrgrOH50Y,Home and Garden,Tools,2,17283
2,vm9pY09Z8CY,Home and Garden,Home Improvements and Repairs,56,74717
3,Q-MmgD4454w,Home and Garden,Home Improvements and Repairs,96,96323
4,a7Ca8u39cW4,Holidays and Traditions,Halloween,92,65133
...,...,...,...,...,...
337,7l51yoO9WPU,Home and Garden,Housekeeping,60,21712
338,0L4gnVIKoNo,Cars & Other Vehicles,Driving Techniques,174,2303
339,pb2BoWwDquM,Home and Garden,Landscaping and Outdoor Building,116,13266
340,iW2Q7lIxrRo,Cars & Other Vehicles,Cars,24,5577


### Clip generation (scene detection)

In [6]:
def color_histogram(frame_bgr):
    """Return a normalized HSV hue/saturation histogram describing a frame's colours."""
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist


def make_clip(video_id, clip_index, frames, times):
    """Build one clip dict, sampling FRAMES_PER_CLIP evenly spaced keyframes from it."""
    n = min(FRAMES_PER_CLIP, len(frames))
    picks = np.linspace(0, len(frames) - 1, n).round().astype(int)
    return {"video_id": video_id, "clip_index": clip_index,
            "start_time": round(times[0], 1), "end_time": round(times[-1], 1),
            "keyframes": [frames[i] for i in picks]}


def generate_clips(video_path, video_id):
    """Split a video into clips, starting a new clip at colour-histogram scene changes.

    We scan the video at ~SCAN_FPS frames per second (neighbouring frames look
    almost identical, so this is enough and keeps it fast on CPU). A clip ends when
    the histogram changes a lot (a scene cut) or it has grown longer than
    MAX_CLIP_SECONDS. Returns a list of clip dicts (see `make_clip`).
    """
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0      # video frame rate (fallback 25)
    step = max(1, round(fps / SCAN_FPS))         # e.g. look at every 25th frame at 25 fps

    clips = []
    clip_frames, clip_times = [], []             # frames/timestamps of the current clip
    prev_hist = None
    frame_index = 0

    while True:
        if not cap.grab():                       # advance to the next frame (cheap, no decode)
            break
        if frame_index % step == 0:              # only decode the frames we actually look at
            ok, frame_bgr = cap.retrieve()       # OpenCV decodes frames as BGR
            if not ok:
                break
            hist = color_histogram(frame_bgr)
            t = frame_index / fps                # timestamp of this frame in seconds

            # detect a scene cut by comparing this frame's colours to the previous one
            scene_change = False
            if prev_hist is not None:
                dist = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_BHATTACHARYYA)
                scene_change = dist > SCENE_THRESHOLD
            too_long = clip_times and (t - clip_times[0]) >= MAX_CLIP_SECONDS

            # close the current clip and start a new one at a cut or the length limit
            if (scene_change or too_long) and clip_times:
                clips.append(make_clip(video_id, len(clips), clip_frames, clip_times))
                clip_frames, clip_times = [], []

            clip_frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))   # store as RGB
            clip_times.append(t)
            prev_hist = hist
        frame_index += 1

    if clip_times:                               # don't forget the final clip
        clips.append(make_clip(video_id, len(clips), clip_frames, clip_times))
    cap.release()
    return clips

### CLIP embeddings

In [9]:
def load_clip_model(name, device):
    """Load a frozen CLIP model and its processor for inference."""
    model = CLIPModel.from_pretrained(name).to(device).eval()
    processor = CLIPProcessor.from_pretrained(name)
    return model, processor

def encode_frames(model, processor, frames, device):
    """Encode a list of RGB frames with CLIP's image encoder into normalized embeddings."""
    images = [Image.fromarray(f) for f in frames]
    inputs = processor(images=images, return_tensors="pt").to(device)
    with torch.no_grad():                                  # frozen model, no gradients
        feats = model.get_image_features(**inputs)
    if not torch.is_tensor(feats):                         # newer transformers wraps the output
        feats = feats.pooler_output
    feats = torch.nn.functional.normalize(feats, p=2, dim=1)   # -> cosine similarity later
    return feats.cpu().numpy()

def embed_clip(model, processor, keyframes, device):
    """Average a clip's keyframe embeddings into a single normalized vector."""
    feats = encode_frames(model, processor, keyframes, device)
    mean = feats.mean(axis=0)
    mean = mean / np.linalg.norm(mean)                     # re-normalize after averaging
    return mean.astype("float32")


def encode_clip_text(model, processor, text, device):
    """Encode a text query with CLIP's text encoder into a normalized embedding."""
    inputs = processor(text=[text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        feats = model.get_text_features(**inputs)
    if not torch.is_tensor(feats):
        feats = feats.pooler_output
    feats = torch.nn.functional.normalize(feats, p=2, dim=1)
    return feats.cpu().numpy().astype("float32")


# load CLIP once
clip_model, clip_processor = load_clip_model(CLIP_MODEL_NAME, device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

c:\Users\marce\anaconda3\envs\applied_ml\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marce\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

### Build and save the clip index

In [13]:
def load_videos(bundle_dir, video_dir, limit=None):
    """Join available_videos.csv with task_ids.csv and keep only videos present on disk."""
    available = pd.read_csv(f"{bundle_dir}/available_videos.csv")
    names = pd.read_csv(f"{bundle_dir}/task_ids.csv", sep="\t",
                        header=None, names=["task_id", "task_name"])
    table = available.merge(names, on="task_id", how="left")

    videos = []
    for _, row in table.iterrows():
        path = f"{video_dir}/{row['video_id']}.mp4"
        if os.path.exists(path):
            videos.append({"video_id": row["video_id"], "path": path,
                           "task_name": row["task_name"]})
        if limit is not None and len(videos) >= limit:
            break
    return videos


def build_clip_index(videos, model, processor, device):
    """Generate + embed clips for each video and return (faiss_index, metadata_df).

    The metadata row at position i describes the vector at index position i, so a
    retrieved vector maps back to its video and timestamps.
    """
    vectors, metadata = [], []
    for v in videos:
        clips = generate_clips(v["path"], v["video_id"])
        for c in clips:
            vectors.append(embed_clip(model, processor, c["keyframes"], device))
            metadata.append({"video_id": c["video_id"], "clip_index": c["clip_index"],
                             "start_time": c["start_time"], "end_time": c["end_time"],
                             "task_name": v["task_name"]})
        print(f"{v['video_id']}: {len(clips)} clips")

    vectors = np.vstack(vectors).astype("float32")
    index = build_faiss_index(vectors)               # reuse the helper from the WikiHow part
    return index, pd.DataFrame(metadata)

In [14]:
import os
# Small batch (limit=3) to test the pipeline.
videos = load_videos(BUNDLE_DIR, VIDEO_DIR, limit=3)
clip_index, clip_metadata = build_clip_index(videos, clip_model, clip_processor, device)
print("Total clips indexed:", clip_index.ntotal)

# Save the clip index and metadata so they can be reused without recomputing.
faiss.write_index(clip_index, "Data/Embeddings/clip_index.faiss")
clip_metadata.to_csv("Data/HowTo100M/clip_metadata.csv", index=False)

8XaqCvOpKxs: 38 clips
cfdrgrOH50Y: 29 clips
vm9pY09Z8CY: 35 clips
Total clips indexed: 102


In [23]:
import os
# Process all videos
videos = load_videos(BUNDLE_DIR, VIDEO_DIR, limit=None)
clip_index, clip_metadata = build_clip_index(videos, clip_model, clip_processor, device)
print("Total clips indexed:", clip_index.ntotal)
faiss.write_index(clip_index, "Data/Embeddings/clip_index.faiss")
clip_metadata.to_csv("Data/HowTo100M/clip_metadata.csv", index=False)

8XaqCvOpKxs: 38 clips
cfdrgrOH50Y: 29 clips
vm9pY09Z8CY: 35 clips
Q-MmgD4454w: 58 clips
a7Ca8u39cW4: 78 clips
vQA0c3HFv1I: 63 clips
ERgiMQLNUtk: 24 clips
OcjHjrvktEo: 19 clips
mlTGzEZPxZk: 11 clips
BRd9l7CUnCE: 46 clips
Wa2wDMeZmlQ: 31 clips
Pat8O9FmDrs: 28 clips
4-8PDaGDsLY: 56 clips
UMXebarGRLU: 26 clips
4us2SfC_IkM: 39 clips
ZfoKmWLu_4g: 12 clips
g1BQxPioCmE: 29 clips
S6aYTYr2OXg: 25 clips
ROX2PVwiOrQ: 10 clips
Ogdx8Z9EwQ8: 14 clips
BEjXku4-HYA: 31 clips
449X0inWQns: 6 clips
gUChmNfpSb8: 32 clips
y8KMt_6Pa1g: 34 clips
cB7Zs93RdX4: 14 clips
xA3Q_M3jLxI: 57 clips
1577jS5nDhQ: 36 clips
23j1q6AeStk: 37 clips
ODt8y5ryLW8: 24 clips
nTDx3sN2dhU: 93 clips
XLDNoK_WAhM: 12 clips
xsAUF6Lz-r8: 16 clips
JFm3TYaloec: 10 clips
2l4AfZZp6SI: 39 clips
PTkdnE1v38I: 53 clips
xHQFdUmouQM: 32 clips
y7C--Cv4gPw: 47 clips
9gDFll6Ge7g: 38 clips
QWMJubLqnRw: 44 clips
PY6INtvS-hY: 33 clips
I6absHmJOOA: 11 clips
6fHJDOWV0M4: 27 clips
BkdggjDxV2U: 26 clips
-7ZkFW7SZTQ: 44 clips
t871adsCxNE: 20 clips
FKduhyuh5UE

In [ ]:
# To reload a saved index/metadata later without recomputing:
clip_index    = faiss.read_index("Data/clip_index.faiss")
clip_metadata = pd.read_csv("Data/clip_metadata.csv")

### Text-to-clip retrieval test

Store all clip embeddings in a retrieval-ready index structure (e.g., FAISS or a comparable light-
weight format). Verify your embeddings with a text-to-clip retrieval test: encode a query with the
CLIP text encoder and return the nearest clips.

In [ ]:
def search_clips(model, processor, index, metadata, device, query, k=5):
    """Embed a text query, find its nearest video clips, and print the matching rows."""
    query_vec = encode_clip_text(model, processor, query, device)
    scores, positions = index.search(query_vec, k=k)
    for pos, score in zip(positions[0], scores[0]):
        row = metadata.iloc[pos]
        print(f"score={score:.4f}  video={row['video_id']}  "
              f"t=[{row['start_time']}-{row['end_time']}s]  task={row['task_name']}")

search_clips(clip_model, clip_processor, clip_index, clip_metadata, device, "cutting a sheet of glass")

score=0.3416  video=cfdrgrOH50Y  t=[226.0-240.0s]  task=Cut Glass
score=0.3415  video=cfdrgrOH50Y  t=[224.0-225.0s]  task=Cut Glass
score=0.3303  video=cfdrgrOH50Y  t=[257.0-271.0s]  task=Cut Glass
score=0.3257  video=cfdrgrOH50Y  t=[272.0-286.0s]  task=Cut Glass
score=0.3246  video=cfdrgrOH50Y  t=[287.0-301.0s]  task=Cut Glass
